# Corrigé du TP2 : nettoyage d'un jeu de données

## Objectifs

- sélectionner des lignes et des colonnes avec `loc`, `iloc`, des filtres booléens et `query`
- créer et transformer des colonnes : calculs vectorisés, `np.where`, `assign`, `map`, `apply`
- nettoyer des colonnes texte avec l'accesseur `.str`
- corriger les types : nombres, entiers nullables, dates, catégories
- traiter les valeurs manquantes, les valeurs aberrantes et les doublons
- trier, renommer les colonnes, gérer l'index
- appliquer une démarche complète de nettoyage au jeu de données des ventes, puis l'exporter en CSV et en Parquet

## Livrable

- un jeu de données propre, exporté en CSV et en Parquet, avec une comparaison de la taille des deux fichiers
- une courte note, en fin de notebook, listant les choix de nettoyage effectués

## Comment travailler

- Le TP suit le cours : chaque partie se fait juste après la section du cours correspondante. Les parties 1 à 6 sont des exercices courts, la partie 7 est le cas pratique : le nettoyage complet des ventes du TP1.
- Complète les cellules marquées `# TODO`, en utilisant les noms de variables indiqués dans l'énoncé.
- Quand un **résultat attendu** est indiqué, compare-le avec ce que tu obtiens.
- Les cellules qui commencent par `# Vérification` contrôlent ton travail avec `assert` : exécute-les sans les modifier. Si tout est juste, elles affichent `OK` ; sinon, elles s'arrêtent sur une `AssertionError` dont le message indique ce qui ne va pas.
- Les questions **Pour aller plus loin** sont facultatives : fais-les si tu as terminé une partie en avance.

## Partie 0 : environnement

Deux possibilités :

- réutiliser l'environnement du TP1 : depuis le dossier de ce TP, active-le avec `source ../tp01-environnement-pandas/.venv/bin/activate`, puis lance `jupyter lab` ;
- ou recréer un environnement dans ce dossier à partir du `requirements.txt` du TP1 :
  ```bash
  python3 -m venv .venv
  source .venv/bin/activate
  python -m pip install -r ../tp01-environnement-pandas/requirements.txt
  jupyter lab
  ```

Le fichier `ressources/ventes.csv` est le fichier brut : ne le modifie jamais. Toutes les corrections se font sur une copie en mémoire, et le résultat est écrit dans de nouveaux fichiers.

In [74]:
import numpy as np
import pandas as pd

brut = pd.read_csv("ressources/ventes.csv")
df = brut.copy()
df.head()


,id_commande,date,region,categorie,produit,Prix unitaire,quantite,Client fidèle
0,1021,2025-01-01,Nouvelle-Aquitaine,Bureau,Lampe LED,268.78,9,False
1,1035,2025-01-02,Bretagne,Bureau,Rangement papier,316.64,3,False
2,1044,2025-01-05,Hauts-de-France,Électroménager,Grille-pain,248.61,10,False
3,1026,2025-01-11,Occitanie,Électroménager,Cafetière,92.78,2,False
4,1053,2025-01-13,Bretagne,Bureau,Bureau assis-debout,341.86,3,True


## Partie 1 : sélectionner des données

*À faire après la section 1 du cours : sélectionner des données.*

On travaille sur `df`, la copie du fichier brut, avec ses noms de colonnes d'origine.

### Question 1.1

Affiche les colonnes `produit` et `quantite` des lignes de labels 10 à 14 avec `loc`. Obtiens ensuite exactement le même résultat avec `iloc`. Pourquoi les bornes ne sont-elles pas les mêmes ?

**Résultat attendu** :

```text
                produit  quantite
10          Stylo plume        10
11      Souris sans fil        10
12  Bureau assis-debout         6
13          Stylo plume         5
14          Grille-pain         9
                produit  quantite
10          Stylo plume        10
11      Souris sans fil        10
12  Bureau assis-debout         6
13          Stylo plume         5
14          Grille-pain         9
```

In [75]:
display(df.loc[10:14, ["produit", "quantite"]])
df.iloc[10:15, [4, 6]]


,produit,quantite
10,Stylo plume,10
11,Souris sans fil,10
12,Bureau assis-debout,6
13,Stylo plume,5
14,Grille-pain,9


,produit,quantite
10,Stylo plume,10
11,Souris sans fil,10
12,Bureau assis-debout,6
13,Stylo plume,5
14,Grille-pain,9


**Réponse** : `loc` sélectionne par label et inclut la borne de fin : `10:14` donne 5 lignes. `iloc` sélectionne par position et exclut la borne de fin : il faut écrire `10:15`. Les colonnes, elles, se désignent par leur nom avec `loc` et par leur position avec `iloc` (`produit` est en position 4, `quantite` en position 6).

### Question 1.2

Sélectionne dans `bureau_5` les commandes de la catégorie `Bureau` dont la quantité est supérieure ou égale à 5. Affiche leur nombre, puis leurs colonnes `id_commande`, `produit` et `quantite`.

**Résultat attendu** :

```text
11
    id_commande              produit  quantite
0          1021            Lampe LED         9
9          1027  Bureau assis-debout        12
12         1005  Bureau assis-debout         6
19         1050   Chaise ergonomique        10
22         1029     Rangement papier         8
38         1041            Lampe LED         7
39         1040   Chaise ergonomique         7
40         1040   Chaise ergonomique         7
47         1025            Lampe LED        11
55         1014  Bureau assis-debout         9
61         1020   Chaise ergonomique         5
```

In [76]:
bureau_5 = df[(df["categorie"] == "Bureau") & (df["quantite"] >= 5)]
print(len(bureau_5))
bureau_5[["id_commande", "produit", "quantite"]]


11


,id_commande,produit,quantite
0,1021,Lampe LED,9
9,1027,Bureau assis-debout,12
12,1005,Bureau assis-debout,6
19,1050,Chaise ergonomique,10
22,1029,Rangement papier,8
38,1041,Lampe LED,7
39,1040,Chaise ergonomique,7
40,1040,Chaise ergonomique,7
47,1025,Lampe LED,11
55,1014,Bureau assis-debout,9


In [77]:
# Vérification
assert len(bureau_5) == 11, "11 commandes attendues : vérifie les deux conditions et les parenthèses"
assert (bureau_5["categorie"] == "Bureau").all() and (bureau_5["quantite"] >= 5).all()
print("Question 1.2 : OK")


Question 1.2 : OK


### Question 1.3

Compte les commandes passées en Bretagne ou en Occitanie avec `isin`. Compare avec ce qu'indique `value_counts` sur la colonne `region` : que manque-t-il au filtre, et pourquoi ?

**Résultat attendu** :

```text
Bretagne ou Occitanie : 12
region
Hauts-de-France         18
Nouvelle-Aquitaine      10
Île-de-France            9
Occitanie                8
Auvergne-Rhône-Alpes     7
Bretagne                 4
NaN                      3
Bretagne                 1
occitanie                1
Ile-de-France            1
Name: count, dtype: int64
```

In [78]:
print("Bretagne ou Occitanie :", df["region"].isin(["Bretagne", "Occitanie"]).sum())
df["region"].value_counts(dropna=False)


Bretagne ou Occitanie : 12


region
Hauts-de-France         18
Nouvelle-Aquitaine      10
Île-de-France            9
Occitanie                8
Auvergne-Rhône-Alpes     7
Bretagne                 4
NaN                      3
Bretagne                 1
occitanie                1
Ile-de-France            1
Name: count, dtype: int64

**Réponse** : Le filtre trouve 12 commandes (4 en Bretagne, 8 en Occitanie). Il manque `Bretagne ` avec une espace finale et `occitanie` en minuscules : `isin` compare des valeurs exactes. Il faudra nettoyer le texte avant de filtrer.

### Question 1.4

En une seule instruction avec `loc`, affiche la date, le produit et la quantité des commandes de plus de 10 unités, triées de la plus grosse à la plus petite.

**Résultat attendu** :

```text
          date              produit  quantite
24  2025-04-18      Écran 24 pouces       480
9   2025-01-31  Bureau assis-debout        12
34  2025-05-15            Cafetière        12
36  2025-05-17            Agrafeuse        12
59  2025-09-07            Agrafeuse        12
26  2025-04-23           Bouilloire        11
44  2025-07-05            Cafetière        11
47  2025-07-11            Lampe LED        11
50  2025-08-03         Casque audio        11
```

In [79]:
df.loc[df["quantite"] > 10, ["date", "produit", "quantite"]].sort_values("quantite", ascending=False)


,date,produit,quantite
24,2025-04-18,Écran 24 pouces,480
9,2025-01-31,Bureau assis-debout,12
34,2025-05-15,Cafetière,12
36,2025-05-17,Agrafeuse,12
59,2025-09-07,Agrafeuse,12
26,2025-04-23,Bouilloire,11
44,2025-07-05,Cafetière,11
47,2025-07-11,Lampe LED,11
50,2025-08-03,Casque audio,11


### Question 1.5

Combien de commandes de clients non fidèles ont un prix unitaire compris entre 100 et 200 euros, bornes incluses ? Utilise `~` et `between`, et range le nombre dans `nb_commandes`.

**Résultat attendu** :

```text
10
```

In [80]:
nb_commandes = (~df["Client fidèle"] & df["Prix unitaire"].between(100, 200)).sum()
print(nb_commandes)


10


### Question 1.6

Avec `query`, sélectionne les commandes de clients fidèles d'au moins 10 unités, et affiche leurs colonnes `id_commande`, `produit` et `quantite`. Attention : le nom de colonne `Client fidèle` contient une espace et doit être entouré d'accents graves.

**Résultat attendu** :

```text
    id_commande              produit  quantite
9          1027  Bureau assis-debout        12
36         1054            Agrafeuse        12
50         1003         Casque audio        11
59         1036            Agrafeuse        12
```

In [81]:
df.query("`Client fidèle` and quantite >= 10")[["id_commande", "produit", "quantite"]]


,id_commande,produit,quantite
9,1027,Bureau assis-debout,12
36,1054,Agrafeuse,12
50,1003,Casque audio,11
59,1036,Agrafeuse,12


### Question 1.7

Crée une copie `essai` de `df` avec `.copy()`. Dans `essai`, remplace par 0 les quantités négatives, en une seule instruction avec `loc`. Affiche ensuite le nombre de quantités négatives dans `essai`, puis dans `df`.

**Résultat attendu** :

```text
0 1
```

In [82]:
essai = df.copy()
essai.loc[essai["quantite"] < 0, "quantite"] = 0
print((essai["quantite"] < 0).sum(), (df["quantite"] < 0).sum())


0 1


In [83]:
# Vérification
assert (essai["quantite"] < 0).sum() == 0, "il reste des quantités négatives dans essai"
assert (df["quantite"] < 0).sum() == 1, "df ne doit pas être modifié : travaille sur la copie essai"
assert len(essai) == len(df), "aucune ligne ne doit être supprimée"
print("Question 1.7 : OK")


Question 1.7 : OK


### Pour aller plus loin 1.8

Exécute l'instruction `essai[essai["quantite"] > 100]["quantite"] = 0`, puis affiche la quantité maximale de `essai`. La quantité de 480 a-t-elle été modifiée ? Écris ensuite la bonne version.

In [84]:
essai[essai["quantite"] > 100]["quantite"] = 0
print(essai["quantite"].max())

essai.loc[essai["quantite"] > 100, "quantite"] = 0
print(essai["quantite"].max())


480
12


/var/folders/z6/kz74j4r96llbbc63nsb2zqsh0000gn/T/ipykernel_76702/3167100436.py:1: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment.
Such chained assignment never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

Try using '.loc[row_indexer, col_indexer] = value' instead, to perform the assignment in a single step.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html#chained-assignment
  essai[essai["quantite"] > 100]["quantite"] = 0


**Réponse** : Non : la première sélection crée une copie temporaire, et l'affectation modifie cette copie, aussitôt perdue. pandas l'annonce par un avertissement `ChainedAssignmentError`. Pour modifier le DataFrame, on sélectionne et on affecte en une seule étape avec `loc`.

## Partie 2 : transformer les données

*À faire après la section 2 du cours : transformer les données.*

Les exercices de cette partie portent sur un petit tableau de commandes, propre, pour se concentrer sur les transformations. Exécute la cellule suivante.

In [85]:
commandes = pd.DataFrame({
    "produit": ["Clavier mécanique", "Souris sans fil", "Écran 24 pouces", "Lampe LED", "Écran 27 pouces", "Stylo plume"],
    "categorie": ["Informatique", "Informatique", "Informatique", "Bureau", "Informatique", "Papeterie"],
    "prix_unitaire": [89.0, 29.9, 189.0, 34.9, 249.0, 24.9],
    "quantite": [2, 10, 1, 4, 3, 12],
})
commandes


,produit,categorie,prix_unitaire,quantite
0,Clavier mécanique,Informatique,89.0,2
1,Souris sans fil,Informatique,29.9,10
2,Écran 24 pouces,Informatique,189.0,1
3,Lampe LED,Bureau,34.9,4
4,Écran 27 pouces,Informatique,249.0,3
5,Stylo plume,Papeterie,24.9,12


### Question 2.1

Ajoute à `commandes` une colonne `montant`, égale au prix unitaire multiplié par la quantité, et une colonne `taille` qui vaut `grosse` si la quantité est supérieure ou égale à 10 et `standard` sinon, avec `np.where`. Affiche les colonnes `produit`, `montant` et `taille`.

**Résultat attendu** :

```text
             produit  montant    taille
0  Clavier mécanique    178.0  standard
1    Souris sans fil    299.0    grosse
2    Écran 24 pouces    189.0  standard
3          Lampe LED    139.6  standard
4    Écran 27 pouces    747.0  standard
5        Stylo plume    298.8    grosse
```

In [145]:
commandes["montant"] = commandes["prix_unitaire"] * commandes["quantite"]
commandes["taille"] = np.where(commandes["quantite"] >= 10, "grosse", "standard")
commandes["remise"] = np.where(commandes["montant"] >= 200, 0.1 * commandes["montant"], 0)
commandes[["produit", "montant", "taille", "remise"]]


,produit,montant,taille,remise
0,Clavier mécanique,178.0,standard,0.00
1,Souris sans fil,299.0,grosse,29.90
2,Écran 24 pouces,189.0,standard,0.00
3,Lampe LED,139.6,standard,0.00
4,Écran 27 pouces,747.0,standard,74.70
5,Stylo plume,298.8,grosse,29.88


In [87]:
# Vérification
assert round(commandes["montant"].sum(), 2) == 1851.4, "vérifie le calcul du montant"
assert (commandes["taille"] == "grosse").sum() == 2, "deux commandes ont au moins 10 unités"
print("Question 2.1 : OK")


Question 2.1 : OK


### Question 2.2

Avec `assign`, crée un nouveau DataFrame `avec_ttc` qui ajoute à `commandes` une colonne `montant_ttc` (montant multiplié par 1,2, arrondi à 2 décimales), sans modifier `commandes`. Supprime ensuite la colonne `taille` de `avec_ttc` avec `drop`. Affiche les colonnes de `avec_ttc`.

**Résultat attendu** :

```text
['produit', 'categorie', 'prix_unitaire', 'quantite', 'montant', 'montant_ttc']
```

In [88]:
avec_ttc = commandes.assign(montant_ttc=(commandes["montant"] * 1.2).round(2))
avec_ttc = avec_ttc.drop(columns=["taille"])
avec_ttc.columns.tolist()


['produit', 'categorie', 'prix_unitaire', 'quantite', 'montant', 'montant_ttc']

In [89]:
# Vérification
assert "montant_ttc" in avec_ttc.columns and "montant_ttc" not in commandes.columns, "assign ne doit pas modifier commandes"
assert "taille" not in avec_ttc.columns and "taille" in commandes.columns, "drop renvoie un nouveau DataFrame : réaffecte le résultat"
print("Question 2.2 : OK")


Question 2.2 : OK


### Question 2.3

1. Avec `map` et le dictionnaire `codes`, ajoute à `commandes` une colonne `code_categorie`.
2. Avec `map` et une fonction `lambda`, ajoute une colonne `type` qui contient le premier mot du nom du produit (`Clavier`, `Souris`...).

Affiche les colonnes `produit`, `code_categorie` et `type`.

**Résultat attendu** :

```text
             produit code_categorie     type
0  Clavier mécanique            INF  Clavier
1    Souris sans fil            INF   Souris
2    Écran 24 pouces            INF    Écran
3          Lampe LED            BUR    Lampe
4    Écran 27 pouces            INF    Écran
5        Stylo plume            PAP    Stylo
```

In [ ]:
codes = {"Bureau": "BUR", "Papeterie": "PAP", "Informatique": "INF", "Électroménager": "ELM"}

commandes["code_categorie"] = commandes["categorie"].map(codes)
commandes["type"] = commandes["produit"].map(lambda p: p.split()[0])
commandes[["produit", "code_categorie", "type"]]


,produit,code_categorie,type
0,Clavier mécanique,INF,Clavier
1,Souris sans fil,INF,Souris
2,Écran 24 pouces,INF,Écran
3,Lampe LED,BUR,Lampe
4,Écran 27 pouces,INF,Écran
5,Stylo plume,PAP,Stylo


In [91]:
# Vérification
assert commandes["code_categorie"].tolist() == ["INF", "INF", "INF", "BUR", "INF", "PAP"]
assert commandes["type"].tolist() == ["Clavier", "Souris", "Écran", "Lampe", "Écran", "Stylo"]
print("Question 2.3 : OK")


Question 2.3 : OK


### Question 2.4

Écris une fonction `libelle(ligne)` qui renvoie un texte de la forme `Clavier mécanique x2`, et applique-la à chaque ligne de `commandes` avec `apply(..., axis=1)`, dans une colonne `libelle`.

Obtiens ensuite le même résultat sans `apply`, par une opération vectorisée, dans une colonne `libelle_vectorise`. Affiche les deux colonnes.

**Résultat attendu** :

```text
                libelle     libelle_vectorise
0  Clavier mécanique x2  Clavier mécanique x2
1   Souris sans fil x10   Souris sans fil x10
2    Écran 24 pouces x1    Écran 24 pouces x1
3          Lampe LED x4          Lampe LED x4
4    Écran 27 pouces x3    Écran 27 pouces x3
5       Stylo plume x12       Stylo plume x12
```

In [92]:
def libelle(ligne):
    return f"{ligne['produit']} x{ligne['quantite']}"


commandes["libelle"] = commandes.apply(libelle, axis=1)
commandes["libelle_vectorise"] = commandes["produit"] + " x" + commandes["quantite"].astype(str)
commandes[["libelle", "libelle_vectorise"]]


,libelle,libelle_vectorise
0,Clavier mécanique x2,Clavier mécanique x2
1,Souris sans fil x10,Souris sans fil x10
2,Écran 24 pouces x1,Écran 24 pouces x1
3,Lampe LED x4,Lampe LED x4
4,Écran 27 pouces x3,Écran 27 pouces x3
5,Stylo plume x12,Stylo plume x12


In [93]:
# Vérification
assert commandes.loc[0, "libelle"] == "Clavier mécanique x2", "le libellé doit être de la forme Clavier mécanique x2"
assert (commandes["libelle"] == commandes["libelle_vectorise"]).all(), "les deux colonnes doivent être identiques"
print("Question 2.4 : OK")


Question 2.4 : OK


### Pour aller plus loin 2.5

Avec `pd.cut`, ajoute à `commandes` une colonne `gamme` qui range chaque produit selon son prix unitaire : `petit prix` jusqu'à 30 euros, `prix moyen` de plus de 30 à 100 euros, `haut de gamme` au-delà, jusqu'à 500 euros. Affiche `produit`, `prix_unitaire` et `gamme`, puis le nombre de produits par gamme.

**Résultat attendu** :

```text
             produit  prix_unitaire          gamme
0  Clavier mécanique           89.0     prix moyen
1    Souris sans fil           29.9     petit prix
2    Écran 24 pouces          189.0  haut de gamme
3          Lampe LED           34.9     prix moyen
4    Écran 27 pouces          249.0  haut de gamme
5        Stylo plume           24.9     petit prix
gamme
petit prix       2
prix moyen       2
haut de gamme    2
Name: count, dtype: int64
```

In [94]:
commandes["gamme"] = pd.cut(
    commandes["prix_unitaire"],
    bins=[0, 30, 100, 500],
    labels=["petit prix", "prix moyen", "haut de gamme"],
)
print(commandes[["produit", "prix_unitaire", "gamme"]])
commandes["gamme"].value_counts()


             produit  prix_unitaire          gamme
0  Clavier mécanique           89.0     prix moyen
1    Souris sans fil           29.9     petit prix
2    Écran 24 pouces          189.0  haut de gamme
3          Lampe LED           34.9     prix moyen
4    Écran 27 pouces          249.0  haut de gamme
5        Stylo plume           24.9     petit prix


gamme
petit prix       2
prix moyen       2
haut de gamme    2
Name: count, dtype: int64

## Partie 3 : nettoyer le texte

*À faire après la section 3 du cours : nettoyer le texte avec l'accesseur `.str`.*

### Question 3.1

Nettoie la Series `villes` dans une nouvelle Series `villes_propres` : retire les espaces superflus, puis mets une majuscule au début de chaque mot avec `str.title`. Que devient la valeur manquante ?

**Résultat attendu** :

```text
0            Paris
1             Lyon
2    Saint-Étienne
3        Marseille
4            Lille
5              NaN
dtype: str
```

In [95]:
villes = pd.Series([" Paris", "LYON ", "saint-étienne", "Marseille", "  lille  ", None])

villes_propres = villes.str.strip().str.title()
villes_propres


0            Paris
1             Lyon
2    Saint-Étienne
3        Marseille
4            Lille
5              NaN
dtype: str

In [96]:
# Vérification
assert villes_propres.dropna().tolist() == ["Paris", "Lyon", "Saint-Étienne", "Marseille", "Lille"]
assert villes_propres.isna().sum() == 1, "la valeur manquante doit rester manquante"
print("Question 3.1 : OK")


Question 3.1 : OK


**Réponse** : Elle reste manquante : les méthodes de l'accesseur `.str` laissent passer les valeurs manquantes sans erreur.

### Question 3.2

Dans `df`, combien de commandes portent sur un produit dont le nom contient `souris` ? Et `écran` ? Ne tiens pas compte des majuscules.

**Résultat attendu** :

```text
10
3
```

In [97]:
print(df["produit"].str.contains("souris", case=False).sum())
print(df["produit"].str.contains("écran", case=False).sum())


10
3


### Question 3.3

1. Dans `numeros`, extrais avec `str.extract` le numéro de chaque référence (les chiffres qui suivent `REF-`), converti en entier.
2. Dans `decoupe`, découpe chaque référence en deux colonnes `code` et `libelle`, sur le séparateur ` / ` (`str.split` avec `expand=True`).

**Résultat attendu** :

```text
[1, 42, 107]
      code            libelle
0  REF-001  Clavier mécanique
1  REF-042    Souris sans fil
2  REF-107    Écran 24 pouces
```

In [98]:
references = pd.Series(["REF-001 / Clavier mécanique", "REF-042 / Souris sans fil", "REF-107 / Écran 24 pouces"])

numeros = references.str.extract(r"REF-(\d+)", expand=False).astype(int)
decoupe = references.str.split(" / ", expand=True)
decoupe.columns = ["code", "libelle"]
print(numeros.tolist())
decoupe


[1, 42, 107]


,code,libelle
0,REF-001,Clavier mécanique
1,REF-042,Souris sans fil
2,REF-107,Écran 24 pouces


In [99]:
# Vérification
assert numeros.tolist() == [1, 42, 107], "numeros doit contenir des entiers"
assert decoupe["libelle"].tolist() == ["Clavier mécanique", "Souris sans fil", "Écran 24 pouces"]
print("Question 3.3 : OK")


Question 3.3 : OK


### Pour aller plus loin 3.4

1. Les numéros de téléphone ci-dessous sont écrits de plusieurs façons. Harmonise-les dans `telephones_propres` en retirant tout ce qui n'est pas un chiffre, avec une expression régulière.
2. Compare `regions.replace("Bretagne", "BZH")` et `regions.str.replace("Bretagne", "BZH")`. Quelle est la différence ?

**Résultat attendu** :

```text
['0123456789', '0123456789', '0123456789', '0123456789']
['BZH', 'Bretagne Sud']
['BZH', 'BZH Sud']
```

In [100]:
telephones = pd.Series(["01 23 45 67 89", "01.23.45.67.89", "0123456789", "01-23-45-67-89"])
regions = pd.Series(["Bretagne", "Bretagne Sud"])

telephones_propres = telephones.str.replace(r"\D", "", regex=True)
print(telephones_propres.tolist())
print(regions.replace("Bretagne", "BZH").tolist())
print(regions.str.replace("Bretagne", "BZH").tolist())


['0123456789', '0123456789', '0123456789', '0123456789']
['BZH', 'Bretagne Sud']
['BZH', 'BZH Sud']


In [101]:
# Vérification
assert telephones_propres.nunique() == 1 and telephones_propres[0] == "0123456789"
print("Question 3.4 : OK")


Question 3.4 : OK


**Réponse** : `Series.replace` remplace des valeurs entières : seule la valeur exactement égale à `Bretagne` change. `Series.str.replace` remplace un morceau de texte à l'intérieur de chaque chaîne : `Bretagne Sud` devient aussi `BZH Sud`.

## Partie 4 : gérer les types

*À faire après la section 4 du cours : gérer les types.*

### Question 4.1

Convertis la Series `prix_texte` en nombres dans `prix_nombres` : remplace la virgule décimale par un point, retire les espaces, puis utilise `pd.to_numeric` avec `errors="coerce"`. Combien de valeurs sont manquantes après conversion, et combien l'étaient déjà avant ?

**Résultat attendu** :

```text
[12.5, 12.5, nan, 1200.0, 45.0, nan]
2 1
```

In [152]:
prix_texte = pd.Series(["12.5", "12,5", "abc", "1 200", "45", None])
prix_nombres = pd.to_numeric(prix_texte.str.replace(",", ".").str.replace(" ", ""), errors="coerce")
print(prix_nombres.tolist())
print(prix_nombres.isna().sum(), prix_texte.isna().sum())


[12.5, 12.5, nan, 1200.0, 45.0, nan]
2 1


In [103]:
# Vérification
assert prix_nombres.dtype == "float64", "prix_nombres doit être numérique"
assert prix_nombres.dropna().tolist() == [12.5, 12.5, 1200.0, 45.0]
print("Question 4.1 : OK")


Question 4.1 : OK


**Réponse** : Deux valeurs sont manquantes après conversion : celle qui l'était déjà, et `abc`, qui n'est pas un nombre. `errors="coerce"` transforme les valeurs non convertibles en `NaN` au lieu de provoquer une erreur ; on peut ensuite les compter et décider quoi en faire.

### Question 4.2

Affiche le type de la Series `quantites`. Pourquoi n'est-ce pas un entier ? Convertis-la dans `quantites_entieres` avec le type entier nullable `Int64`, et affiche le résultat.

**Résultat attendu** :

```text
float64
0       3
1    <NA>
2       5
3       2
dtype: Int64
```

In [104]:
quantites = pd.Series([3, None, 5, 2])

print(quantites.dtype)
quantites_entieres = quantites.astype("Int64")
quantites_entieres


float64


0       3
1    <NA>
2       5
3       2
dtype: Int64

In [105]:
# Vérification
assert str(quantites_entieres.dtype) == "Int64", "utilise le type Int64, avec une majuscule"
assert quantites_entieres.isna().sum() == 1
print("Question 4.2 : OK")


Question 4.2 : OK


**Réponse** : Un entier numpy ne peut pas représenter une valeur manquante : pandas passe donc toute la colonne en `float64`. Le type `Int64` garde des entiers et note la valeur manquante `<NA>`.

### Question 4.3

Convertis la Series `dates_texte`, écrite au format français jour/mois/année, en dates dans `dates`, avec `pd.to_datetime`, le bon `format` et `errors="coerce"`. Affiche ensuite le mois et le nom du jour de chaque date avec l'accesseur `.dt`.

**Résultat attendu** :

```text
0   2025-02-03
1   2025-12-31
2   2025-07-15
3          NaT
dtype: datetime64[us]
[2.0, 12.0, 7.0, nan]
['Monday', 'Wednesday', 'Tuesday', nan]
```

In [106]:
dates_texte = pd.Series(["03/02/2025", "31/12/2025", "15/07/2025", "pas de date"])

dates = pd.to_datetime(dates_texte, format="%d/%m/%Y", errors="coerce")
print(dates)
print(dates.dt.month.tolist())
print(dates.dt.day_name().tolist())


0   2025-02-03
1   2025-12-31
2   2025-07-15
3          NaT
dtype: datetime64[us]
[2.0, 12.0, 7.0, nan]
['Monday', 'Wednesday', 'Tuesday', nan]


In [107]:
# Vérification
assert dates.dt.month.tolist()[:3] == [2, 12, 7], "03/02/2025 est le 3 février : vérifie le format"
assert dates.isna().sum() == 1
print("Question 4.3 : OK")


Question 4.3 : OK


## Partie 5 : valeurs manquantes, valeurs aberrantes et doublons

*À faire après la section 5 du cours : valeurs manquantes, valeurs aberrantes et doublons.*

Les relevés ci-dessous viennent d'un magasin : une température et un niveau de stock par jour, avec des trous. Exécute la cellule suivante.

In [108]:
releves = pd.DataFrame({
    "jour": [1, 2, 3, 4, 5, 6, 7, 8],
    "temperature": [8.5, 9.1, None, None, 11.3, 10.8, None, 12.0],
    "stock": [120, None, None, 95, None, 80, 78, None],
    "magasin": ["Lyon", "Lyon", None, "Lyon", "Lyon", None, "Lyon", "Lyon"],
})
releves


,jour,temperature,stock,magasin
0,1,8.5,120.0,Lyon
1,2,9.1,NaN,Lyon
2,3,NaN,NaN,NaN
3,4,NaN,95.0,Lyon
4,5,11.3,NaN,Lyon
5,6,10.8,80.0,NaN
6,7,NaN,78.0,Lyon
7,8,12.0,NaN,Lyon


### Question 5.1

Affiche le nombre de valeurs manquantes par colonne, puis leur proportion arrondie à 2 décimales.

**Résultat attendu** :

```text
jour           0
temperature    3
stock          4
magasin        2
dtype: int64
jour           0.00
temperature    0.38
stock          0.50
magasin        0.25
dtype: float64
```

In [109]:
print(releves.isna().sum())
releves.isna().mean().round(2)


jour           0
temperature    3
stock          4
magasin        2
dtype: int64


jour           0.00
temperature    0.38
stock          0.50
magasin        0.25
dtype: float64

### Question 5.2

Construis `complete`, une copie de `releves` dans laquelle :

- la température manquante est estimée par interpolation entre les jours voisins ;
- le stock manquant est remplacé par la dernière valeur connue ;
- le magasin manquant est remplacé par `Inconnu`.

Pourquoi l'interpolation convient-elle à la température, et la propagation au stock ?

**Résultat attendu** :

```text
   jour  temperature  stock  magasin
0     1     8.500000  120.0     Lyon
1     2     9.100000  120.0     Lyon
2     3     9.833333  120.0  Inconnu
3     4    10.566667   95.0     Lyon
4     5    11.300000   95.0     Lyon
5     6    10.800000   80.0  Inconnu
6     7    11.400000   78.0     Lyon
7     8    12.000000   78.0     Lyon
```

In [110]:
complete = releves.copy()
complete["temperature"] = complete["temperature"].interpolate()
complete["stock"] = complete["stock"].ffill()
complete["magasin"] = complete["magasin"].fillna("Inconnu")
complete


,jour,temperature,stock,magasin
0,1,8.500000,120.0,Lyon
1,2,9.100000,120.0,Lyon
2,3,9.833333,120.0,Inconnu
3,4,10.566667,95.0,Lyon
4,5,11.300000,95.0,Lyon
5,6,10.800000,80.0,Inconnu
6,7,11.400000,78.0,Lyon
7,8,12.000000,78.0,Lyon


In [111]:
# Vérification
assert complete.isna().sum().sum() == 0, "il reste des valeurs manquantes"
assert round(complete.loc[2, "temperature"], 2) == 9.83, "la température du jour 3 doit être interpolée entre 9.1 et 11.3"
assert complete.loc[1, "stock"] == 120, "le stock du jour 2 doit reprendre la valeur du jour 1"
assert releves.isna().sum().sum() == 9, "releves ne doit pas être modifié"
print("Question 5.2 : OK")


Question 5.2 : OK


**Réponse** : La température évolue progressivement d'un jour à l'autre : une valeur intermédiaire entre les deux jours voisins est une bonne estimation. Le stock, lui, ne change que lorsqu'un événement se produit : en l'absence de relevé, la dernière valeur connue reste valable.

### Question 5.3

Combien de lignes de `releves` restent avec `dropna()` sans paramètre ? Et avec `dropna(subset=["temperature"])` ? Pourquoi un tel écart ?

**Résultat attendu** :

```text
1
5
```

In [112]:
print(len(releves.dropna()))
print(len(releves.dropna(subset=["temperature"])))


1
5


**Réponse** : Sans `subset`, `dropna` supprime toute ligne qui contient au moins une valeur manquante, dans n'importe quelle colonne : il ne reste qu'une ligne complète. Avec `subset`, on ne supprime que les lignes où la température manque.

### Question 5.4

Un capteur renvoie `-999` en cas de panne, et a enregistré une mesure impossible de 250 degrés. Dans `capteur_propre`, remplace avec `mask` les mesures hors de l'intervalle de -50 à 60 degrés par des valeurs manquantes, puis remplace ces valeurs manquantes par la médiane des mesures valides. Compare la moyenne avant et après.

**Résultat attendu** :

```text
[12.1, 15.3, 14.2, 14.2, 13.8, 14.2, 16.1]
-96.79 14.27
```

In [113]:
capteur = pd.Series([12.1, 15.3, -999.0, 14.2, 13.8, 250.0, 16.1])

capteur_propre = capteur.mask(~capteur.between(-50, 60))
capteur_propre = capteur_propre.fillna(capteur_propre.median())
print(capteur_propre.tolist())
print(round(capteur.mean(), 2), round(capteur_propre.mean(), 2))


[12.1, 15.3, 14.2, 14.2, 13.8, 14.2, 16.1]
-96.79 14.27


In [114]:
# Vérification
assert capteur_propre.between(-50, 60).all(), "il reste des mesures aberrantes"
assert capteur_propre[2] == 14.2 and capteur_propre[5] == 14.2, "les mesures aberrantes doivent être remplacées par la médiane des mesures valides, 14.2"
print("Question 5.4 : OK")


Question 5.4 : OK


### Question 5.5

Le fichier `clients` ci-dessous contient des doublons.

1. Combien de lignes sont des doublons exacts ?
2. Combien de lignes ont un `id_client` déjà vu, même si le reste diffère ?
3. Affiche toutes les lignes concernées par un doublon d'identifiant (`keep=False`).
4. Le client 4 a déménagé : garde, pour chaque identifiant, la dernière ligne, dans `clients_uniques`.

**Résultat attendu** :

```text
1
2
   id_client     nom   ville
1          2  Martin   Paris
2          2  Martin   Paris
4          4    Roux  Nantes
5          4    Roux    Nice
   id_client     nom  ville
0          1  Durand   Lyon
2          2  Martin  Paris
3          3   Petit  Lille
5          4    Roux   Nice
```

In [115]:
clients = pd.DataFrame({
    "id_client": [1, 2, 2, 3, 4, 4],
    "nom": ["Durand", "Martin", "Martin", "Petit", "Roux", "Roux"],
    "ville": ["Lyon", "Paris", "Paris", "Lille", "Nantes", "Nice"],
})

print(clients.duplicated().sum())
print(clients.duplicated(subset=["id_client"]).sum())
display(clients[clients.duplicated(subset=["id_client"], keep=False)])
clients_uniques = clients.drop_duplicates(subset=["id_client"], keep="last")
clients_uniques


1
2


,id_client,nom,ville
1,2,Martin,Paris
2,2,Martin,Paris
4,4,Roux,Nantes
5,4,Roux,Nice


,id_client,nom,ville
0,1,Durand,Lyon
2,2,Martin,Paris
3,3,Petit,Lille
5,4,Roux,Nice


In [116]:
# Vérification
assert clients_uniques["id_client"].is_unique and len(clients_uniques) == 4
assert clients_uniques.loc[clients_uniques["id_client"] == 4, "ville"].item() == "Nice", "garde la dernière ligne de chaque client"
print("Question 5.5 : OK")


Question 5.5 : OK


## Partie 6 : trier, renommer, gérer l'index

*À faire après la section 6 du cours : trier, renommer, gérer l'index.*

### Question 6.1

Trie `commandes` par catégorie dans l'ordre alphabétique, puis par montant décroissant, et affiche `categorie`, `produit` et `montant`. Affiche ensuite les 3 commandes au plus gros montant avec `nlargest`.

**Résultat attendu** :

```text
      categorie            produit  montant
3        Bureau          Lampe LED    139.6
4  Informatique    Écran 27 pouces    747.0
1  Informatique    Souris sans fil    299.0
2  Informatique    Écran 24 pouces    189.0
0  Informatique  Clavier mécanique    178.0
5     Papeterie        Stylo plume    298.8
           produit  montant
4  Écran 27 pouces    747.0
1  Souris sans fil    299.0
5      Stylo plume    298.8
```

In [117]:
display(commandes.sort_values(["categorie", "montant"], ascending=[True, False])[["categorie", "produit", "montant"]])
commandes.nlargest(3, "montant")[["produit", "montant"]]


,categorie,produit,montant
3,Bureau,Lampe LED,139.6
4,Informatique,Écran 27 pouces,747.0
1,Informatique,Souris sans fil,299.0
2,Informatique,Écran 24 pouces,189.0
0,Informatique,Clavier mécanique,178.0
5,Papeterie,Stylo plume,298.8


,produit,montant
4,Écran 27 pouces,747.0
1,Souris sans fil,299.0
5,Stylo plume,298.8


### Question 6.2

Écris une fonction `harmoniser_colonnes(tableau)` qui renvoie une copie du DataFrame reçu, dont les noms de colonnes sont en *snake_case* : sans espaces superflus, en minuscules, espaces remplacées par des tirets bas, sans accents. Teste-la sur `fichier_clients`, puis sur `brut`.

**Résultat attendu** :

```text
['code_client', 'nom', 'ville', 'date_de_creation', 'montant_ttc']
['id_commande', 'date', 'region', 'categorie', 'produit', 'prix_unitaire', 'quantite', 'client_fidele']
```

In [118]:
fichier_clients = pd.DataFrame(columns=["Code Client", " Nom ", "Ville", "Date de création", "Montant TTC"])


def harmoniser_colonnes(tableau):
    resultat = tableau.copy()
    resultat.columns = (
        resultat.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.normalize("NFKD")
        .str.encode("ascii", errors="ignore")
        .str.decode("ascii")
    )
    return resultat


print(harmoniser_colonnes(fichier_clients).columns.tolist())
print(harmoniser_colonnes(brut).columns.tolist())


['code_client', 'nom', 'ville', 'date_de_creation', 'montant_ttc']
['id_commande', 'date', 'region', 'categorie', 'produit', 'prix_unitaire', 'quantite', 'client_fidele']


In [119]:
# Vérification
assert harmoniser_colonnes(fichier_clients).columns.tolist() == ["code_client", "nom", "ville", "date_de_creation", "montant_ttc"]
assert "Prix unitaire" in brut.columns, "la fonction doit renvoyer une copie, sans modifier le DataFrame reçu"
print("Question 6.2 : OK")


Question 6.2 : OK


### Question 6.3

1. Vérifie que la colonne `produit` de `commandes` est unique, puis fais-en l'index dans un DataFrame `par_produit`.
2. Affiche avec `loc` la quantité de `Lampe LED`.
3. Reviens à un index numérique avec `reset_index` et affiche les colonnes obtenues.
4. Dans `df`, la colonne `id_commande` pourrait-elle servir d'index ?

**Résultat attendu** :

```text
True
4
['produit', 'categorie', 'prix_unitaire']
False
```

In [120]:
print(commandes["produit"].is_unique)
par_produit = commandes.set_index("produit")
print(par_produit.loc["Lampe LED", "quantite"])
print(par_produit.reset_index().columns.tolist()[:3])
print(df["id_commande"].is_unique)


True
4
['produit', 'categorie', 'prix_unitaire']
False


**Réponse** : Pas encore : `id_commande` n'est pas unique dans le fichier brut, à cause des doublons. Il faudra d'abord les supprimer.

### Pour aller plus loin 6.4

Avec `reindex`, réorganise les colonnes de `commandes` dans l'ordre `produit`, `quantite`, `prix_unitaire`, `remise`. Que contient la colonne `remise`, qui n'existait pas ?

**Résultat attendu** :

```text
             produit  quantite  prix_unitaire  remise
0  Clavier mécanique         2           89.0     NaN
1    Souris sans fil        10           29.9     NaN
2    Écran 24 pouces         1          189.0     NaN
3          Lampe LED         4           34.9     NaN
4    Écran 27 pouces         3          249.0     NaN
5        Stylo plume        12           24.9     NaN
```

In [121]:
commandes.reindex(columns=["produit", "quantite", "prix_unitaire", "remise"])


,produit,quantite,prix_unitaire,remise
0,Clavier mécanique,2,89.0,NaN
1,Souris sans fil,10,29.9,NaN
2,Écran 24 pouces,1,189.0,NaN
3,Lampe LED,4,34.9,NaN
4,Écran 27 pouces,3,249.0,NaN
5,Stylo plume,12,24.9,NaN


**Réponse** : Elle est créée et remplie de valeurs manquantes : `reindex` crée les labels absents.